In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install timm streamlit pyngrok opencv-python-headless

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 589.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 27.2 MB/s eta 0:00:00


BACKEND

In [ ]:
%%writefile pcb_backend.py

import cv2
import numpy as np
import torch
import timm
from torchvision import transforms
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# class order must match training dataset
class_names = [
    "Missing_hole",
    "Mouse_bite",
    "Open_circuit",
    "Short",
    "Spur",
    "Spurious_copper"
]

# CNN preprocessing
transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor()
])

# load trained model
model = timm.create_model(
    "efficientnet_b0",
    pretrained=False,
    num_classes=6
)

model.load_state_dict(
    torch.load(
        "/content/drive/MyDrive/PCB_PROJECT/pcb_defect_model.pth",
        map_location=device
    )
)

model = model.to(device)
model.eval()


# ------------------------------------------------
# DEFECT DETECTION USING IMAGE SUBTRACTION
# ------------------------------------------------

def detect_defects(template, test):

    test = cv2.resize(test,(template.shape[1],template.shape[0]))

    template_gray = cv2.cvtColor(template,cv2.COLOR_BGR2GRAY)
    test_gray = cv2.cvtColor(test,cv2.COLOR_BGR2GRAY)

    diff = cv2.absdiff(template_gray,test_gray)

    diff = cv2.GaussianBlur(diff,(5,5),0)

    _,thresh = cv2.threshold(
        diff,
        0,
        255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    kernel = np.ones((5,5),np.uint8)

    # remove noise
    thresh = cv2.morphologyEx(thresh,cv2.MORPH_OPEN,kernel,iterations=2)

    # merge nearby defect pixels (important for SHORT defects)
    thresh = cv2.dilate(thresh,kernel,iterations=2)

    thresh = cv2.morphologyEx(thresh,cv2.MORPH_CLOSE,kernel,iterations=2)

    contours,_ = cv2.findContours(
        thresh,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    boxes = []

    for cnt in contours:

        area = cv2.contourArea(cnt)

        # filter noise
        if area < 200 or area > 10000:
            continue

        x,y,w,h = cv2.boundingRect(cnt)

        boxes.append((x,y,w,h))

    return boxes


# ------------------------------------------------
# CNN CLASSIFICATION
# ------------------------------------------------

def classify_roi(roi):

    img = cv2.resize(roi,(128,128))

    img = cv2.cvtColor(img,cv2.COLOR_BGR2RGB)

    img = Image.fromarray(img)

    img = transform(img)

    img = img.unsqueeze(0).to(device)

    with torch.no_grad():

        outputs = model(img)

        probs = torch.nn.functional.softmax(outputs,dim=1)

        conf,pred = torch.max(probs,1)

    label = class_names[pred.item()]
    confidence = conf.item()

    return label,confidence


# ------------------------------------------------
# COMPLETE PIPELINE
# ------------------------------------------------

def process_pcb(template_path,test_path):

    template = cv2.imread(template_path)
    test = cv2.imread(test_path)

    boxes = detect_defects(template,test)

    output = test.copy()

    results = []

    for (x,y,w,h) in boxes:

        # padded ROI improves classification
        pad = 10

        x1 = max(0,x-pad)
        y1 = max(0,y-pad)
        x2 = min(test.shape[1],x+w+pad)
        y2 = min(test.shape[0],y+h+pad)

        roi = test[y1:y2,x1:x2]

        label,conf = classify_roi(roi)

        # ignore weak predictions
        if conf < 0.85:
            continue

        # ------------------------------------------------
        # SHAPE-BASED CORRECTION (Short vs Open_circuit)
        # ------------------------------------------------

        aspect_ratio = w / float(h)

        if label == "Open_circuit":

            if 0.8 < aspect_ratio < 1.2:
                label = "Short"

        elif label == "Short":

            if aspect_ratio > 2.5 or aspect_ratio < 0.4:
                label = "Open_circuit"

        results.append((label,conf))

        cv2.rectangle(output,(x,y),(x+w,y+h),(0,255,0),2)

        cv2.putText(
            output,
            f"{label} {conf:.2f}",
            (x,y-10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0,255,0),
            2
        )

    return output,results

Writing pcb_backend.py


In [ ]:
%%writefile app.py

import streamlit as st
import tempfile
import pandas as pd
import cv2

from pcb_backend import process_pcb

st.set_page_config(page_title="PCB Defect Detection", layout="wide")

# ---------- HEADER ----------
st.title("🔍 PCB Defect Detection System")
st.caption("Upload template and test images to detect PCB defects")

st.divider()

# ---------- UPLOAD SECTION ----------
col1, col2 = st.columns(2)

with col1:
    template = st.file_uploader("Upload Template Image", type=["jpg", "png"])

with col2:
    test = st.file_uploader("Upload Test Image", type=["jpg", "png"])

st.divider()

# ---------- PROCESS ----------
if template and test:

    # Save temp files
    template_file = tempfile.NamedTemporaryFile(delete=False)
    template_file.write(template.read())

    test_file = tempfile.NamedTemporaryFile(delete=False)
    test_file.write(test.read())

    with st.spinner("Analyzing PCB..."):
        output, results = process_pcb(
            template_file.name,
            test_file.name
        )

    # ---------- RESULT IMAGE ----------
    st.subheader("Detected Output")
    st.image(output, channels="BGR", use_container_width=True)

    # ---------- DEFECTS ----------
    st.subheader("Detected Defects")

    if len(results) == 0:
        st.warning("No defects detected")
    else:
        for r in results:
            st.markdown(f"✔ **{r[0]}** ({r[1]:.2f})")

    # ---------- SUMMARY ----------
    st.subheader("Summary")
    st.markdown(f"**Total Defects:** {len(results)}")

    st.divider()

    # ---------- DOWNLOAD ----------
    cv2.imwrite("result.png", output)

    col1, col2 = st.columns(2)

    with col1:
        with open("result.png", "rb") as f:
            st.download_button("⬇ Download Image", f, "result.png")

    if results:
        df = pd.DataFrame(results, columns=["Defect", "Confidence"])
        csv = df.to_csv(index=False).encode()

        with col2:
            st.download_button("⬇ Download Report", csv, "report.csv")

Writing app.py


START STREAMLIT

In [ ]:
from pyngrok import ngrok
import time

ngrok.kill()

!streamlit run app.py &>/content/logs.txt &

time.sleep(10)

print(ngrok.connect(8501))

NgrokTunnel: "https://criselda-vexed-stuart.ngrok-free.dev" -> "http://localhost:8501"
